# Demand Prediction Model
In this notebook, we will build a machine learning model to predict the next week's demand for each product. We will aggregate the daily demand data into weekly intervals, create historical lag features, and train a Random Forest Regressor to forecast future quantities.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pickle

In [2]:
#load the daily demand and products datasets
daily_demand = pd.read_csv('../data/processed/daily_product_demand.csv')
products_df = pd.read_csv('../data/processed/cleaned_products.csv')

#convert order_date to datetime to ensure proper extraction
daily_demand['order_date'] = pd.to_datetime(daily_demand['order_date'])

print(f"loaded {len(daily_demand)} daily demand records.")

loaded 4908 daily demand records.


## 1. Data Aggregation & Feature Engineering
We need to predict "next week's demand". Therefore, we must aggregate our daily sales data into weekly buckets. After grouping, we will create "lag features" (e.g., how many items sold last week, and the week before) which are critical predictors for time-series forecasting.

In [3]:
#extract year and week for weekly aggregation
daily_demand['year'] = daily_demand['order_date'].dt.isocalendar().year
daily_demand['week'] = daily_demand['order_date'].dt.isocalendar().week

In [4]:
#aggregate to weekly demand per product
weekly_demand = daily_demand.groupby(['product_id', 'year', 'week'])['total_daily_quantity'].sum().reset_index()
weekly_demand.rename(columns={'total_daily_quantity': 'weekly_quantity'}, inplace=True)

In [5]:
#sort values chronologically to ensure lag features calculate correctly
weekly_demand = weekly_demand.sort_values(by=['product_id', 'year', 'week'])

In [6]:
#create lag features for the past 1 and 2 weeks to use as predictors
weekly_demand['lag_1'] = weekly_demand.groupby('product_id')['weekly_quantity'].shift(1)
weekly_demand['lag_2'] = weekly_demand.groupby('product_id')['weekly_quantity'].shift(2)

In [7]:
#drop rows with nan values caused by the shifting operation
model_data = weekly_demand.dropna().copy()
display(model_data.head())

,product_id,year,week,weekly_quantity,lag_1,lag_2
2,1,2024,27,4,3.0,2.0
3,1,2024,28,3,4.0,3.0
4,1,2024,32,3,3.0,4.0
5,1,2024,40,4,3.0,3.0
6,1,2024,44,1,4.0,3.0


## 2. Train-Test Split
For time-series or temporal data, we cannot do a random split. We will train our model on historical data and test it on the most recent week available in our dataset to accurately gauge real-world performance.

In [8]:
#define features (x) and target (y)
X = model_data[['product_id', 'week', 'lag_1', 'lag_2']]
y = model_data['weekly_quantity']

#find the most recent year and week in the dataset
max_year = model_data['year'].max()
max_week = model_data[model_data['year'] == max_year]['week'].max()

In [9]:
#isolate the last week as the test set
test_mask = (model_data['year'] == max_year) & (model_data['week'] == max_week)
X_train, X_test = X[~test_mask], X[test_mask]
y_train, y_test = y[~test_mask], y[test_mask]

print(f"training samples: {len(X_train)}")
print(f"testing samples: {len(X_test)}")

training samples: 4051
testing samples: 13


## 3. Model Training & Evaluation
We will utilize a Random Forest Regressor. It handles non-linear relationships well and is robust against overfitting, making it an excellent choice for baseline demand forecasting.

In [10]:
#initialize and train random forest regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

#predict on the test set
predictions = rf_model.predict(X_test)

#evaluate model performance
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print(f"model evaluation metrics:")
print(f"mean absolute error (mae): {mae:.2f} units")
print(f"root mean squared error (rmse): {rmse:.2f} units")

model evaluation metrics:
mean absolute error (mae): 1.40 units
root mean squared error (rmse): 1.51 units


## 4. Predicting "Next Week's" Demand
To predict the true *next* week for the business, we take the latest known week's data for every single product, shift the lag features forward, and run the model inference. We will then format it exactly as requested in the business requirements.

In [11]:
#isolate the latest known week for each product to forecast the future
latest_data = weekly_demand.sort_values(by=['year', 'week']).groupby('product_id').tail(1).copy()

#shift the current reality into the lag features for the future prediction
latest_data['lag_2'] = latest_data['lag_1']
latest_data['lag_1'] = latest_data['weekly_quantity']

#increment the week indicator for the forecast
latest_data['week'] = latest_data['week'] + 1

#prepare the feature matrix for the future
X_future = latest_data[['product_id', 'week', 'lag_1', 'lag_2']]

#predict future demand and round up to whole units
future_predictions = rf_model.predict(X_future)
latest_data['predicted_demand'] = np.ceil(future_predictions).astype(int)

#merge with products dataframe to get readable names
output_df = latest_data.merge(products_df, on='product_id', how='left')

#format the final output table
final_forecast = output_df[['product_name', 'predicted_demand']].rename(
    columns={'product_name': 'Product', 'predicted_demand': 'Predicted Demand'}
)

#display the top 10 products predicted demand
print("next week's predicted demand:")
display(final_forecast.head(10))

next week's predicted demand:


,Product,Predicted Demand
0,Football,2
1,Sofa,3
2,Shirt,2
3,T-Shirt,4
4,Badminton Racket,3
5,Bluetooth Speaker,3
6,Camera,2
7,Machine Learning Book,4
8,Data Science Guide,3
9,Sofa,3


## 5. Serializing the Model
We will save the trained Random Forest model and the `latest_data` state (since we need the most recent lag features to make live predictions) so they can be loaded into the Streamlit app.

In [12]:
#save the trained random forest model
with open('../models/demand_rf_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

#save the latest state for inference lag features
latest_data[['product_id', 'week', 'lag_1', 'lag_2']].to_csv('../data/processed/latest_demand_features.csv', index=False)

print("demand prediction model and feature states successfully saved.")

demand prediction model and feature states successfully saved.
